# Graph Exploration

Deep dive into the `AnalysisGraph` structure: nodes, edges,
reachability, and call paths. We use `r01/e03` (order overwrite)
which has multiple routes and a richer call graph.

In [1]:
%load_ext autoreload
%autoreload 2
from confusion_sast.notebook import *

## Graph Structure

In [2]:
t = targets()
g = load(t.r01.e03)
g

Methods,Rule,Sources,Keys,Flags
POST,/orders,form,"delivery_address, items",
POST,/cart//items,json,item_id,
POST,/cart//checkout,"form, json",,MULTI-SOURCE
POST,/e2e/balance,json,"balance, user_id",


In [3]:
g.stats()

Metric,Value
nodes,68
edges,117
call_sites,142
routes,10
input_accesses,15
before_requests,0
dict_merges,1
unique_keys,7
unique_sources,3


## Raw NetworkX Data

Under the hood, `g.g` is a `networkx.DiGraph`. Nodes are function
qualnames; edges represent call relationships.

In [4]:
# First 8 nodes with their attributes
list(g.g.nodes(data=True))[:8]

[('routes.index', {'kind': 'endpoint'}),
 ('routes.view_balance', {'kind': 'endpoint'}),
 ('routes.list_menu_items', {'kind': 'endpoint'}),
 ('routes.list_orders', {'kind': 'endpoint'}),
 ('routes.create_new_order', {'kind': 'function'}),
 ('routes.create_new_cart', {'kind': 'endpoint'}),
 ('routes.add_item_to_cart_endpoint', {'kind': 'function'}),
 ('routes.checkout_cart', {'kind': 'function'})]

In [5]:
# First 8 edges
list(g.g.edges(data=True))[:8]

[('routes.index',
  'route',
  {'call_count': 1,
   'callsites': [CallEdge(caller_qualname='routes.index',
                          callee_qualname='route',
                          location=Location(file='/Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py',
                                            line=30,
                                            col=1),
                          argument_map=None)]}),
 ('routes.view_balance',
  'auth.get_authenticated_user',
  {'call_count': 1,
   'callsites': [CallEdge(caller_qualname='routes.view_balance',
                          callee_qualname='auth.get_authenticated_user',
                          location=Location(file='/Users/irina/src/unsafe-code/vulnerabilities/python/flask/confusion/webapp/r01_input_source_confusion/e03_order_overwrite/routes.py',
                                            line=38,
                                            col=11)

## Visual Graph

`GraphView` renders the call graph as a focused navigation surface.
It supports scope switching, text search, layout changes, and
call-site inspection through clickable edges.

In [6]:
gv = GraphView(g, layout="dagre")
gv.widget

In [7]:
# Focus on a single endpoint and its reachable subgraph
gv.focus_endpoint("routes.checkout_cart")
gv.widget

In [8]:
# Endpoint browser for open-ended exploration
et = EndpointTable(g)
et.widget

In [9]:
# Code-path browser for source-grounded graph review
cp = CodePathView(target_path=t.r01.e03.path)
cp.show_function(g, "routes.checkout_cart")
cp.widget

## Call Paths

Trace how a handler reaches a function that accesses user input.

In [10]:
# What functions can create_new_order reach?
reachable = g.reachable_from("routes.create_new_order")
print(f"{len(reachable)} functions reachable")
# Show the ones that have input accesses
for fn in sorted(reachable):
    accs = g.accesses_in(fn)
    if accs:
        print(f"  {fn}: {[a.raw_code for a in accs]}")

30 functions reachable
  routes.create_new_order: ["request.form.get('delivery_address')", 'request.form']
  utils.check_price_and_availability: ["data.getlist('items')"]
  utils.get_order_items: ["data.getlist('items')"]


In [11]:
# Shortest call path from handler to a callee
path = g.call_path("routes.create_new_order", "utils.check_price_and_availability")
print("Path:", path)

path2 = g.call_path("routes.create_new_order", "utils.get_order_items")
print("Path:", path2)

Path: ['routes.create_new_order', 'utils.check_price_and_availability']
Path: ['routes.create_new_order', 'utils.get_order_items']


In [12]:
# Highlight a path in the graph view
gv.reset_highlights()
if path:
    gv.highlight_path(path)
gv.widget

## Input Accesses per Endpoint

Each endpoint handler can reach input accesses in its own body
and in any functions it calls.

In [13]:
for handler, route, accs in g.by_endpoint():
    if not accs:
        continue
    sources = {a.source for a in accs}
    keys = {a.key_literal for a in accs if a.key_literal}
    print(f"{route.handler_name} ({route.rule} {route.methods})")
    print(f"  sources: {sources}")
    print(f"  keys: {keys}")
    print(f"  accesses: {len(accs)}")
    print()

create_new_order (/orders ('POST',))
  sources: {InputSource.FORM}
  keys: {'delivery_address', 'items'}
  accesses: 4

add_item_to_cart_endpoint (/cart/<cart_id>/items ('POST',))
  sources: {InputSource.JSON}
  keys: {'item_id'}
  accesses: 2

checkout_cart (/cart/<cart_id>/checkout ('POST',))
  sources: {InputSource.JSON, InputSource.FORM}
  keys: set()
  accesses: 2

e2e_balance (/e2e/balance ('POST',))
  sources: {InputSource.JSON}
  keys: {'balance', 'user_id'}
  accesses: 3



## Source Code Context

`show_source()` prints the source lines around an input access.

In [14]:
# Show source for each form access
form_accs = accesses(g, source="form")
for a in form_accs[:3]:
    print(f"--- {a.function_qualname}: {a.raw_code} ---")
    show_source(a)
    print()

--- routes.create_new_order: request.form.get('delivery_address') ---



--- routes.create_new_order: request.form ---



--- routes.checkout_cart: request.form ---


## NetworkX Analysis

Since the graph is a standard `networkx.DiGraph`, you can use
any NetworkX algorithm.

In [15]:
import networkx as nx

# Weakly connected components
components = list(nx.weakly_connected_components(g.g))
print(f"{len(components)} weakly connected components")
for i, comp in enumerate(sorted(components, key=len, reverse=True)[:3]):
    print(f"  Component {i}: {len(comp)} nodes")

print()

# Top nodes by out-degree (most callees)
by_degree = sorted(g.g.out_degree(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 by out-degree:")
for node, deg in by_degree:
    print(f"  {node}: {deg}")

1 weakly connected components
  Component 0: 68 nodes

Top 5 by out-degree:
  routes.create_new_order: 9
  routes.checkout_cart: 9
  routes.list_orders: 7
  routes.add_item_to_cart_endpoint: 7
  routes.e2e_balance: 7


## Comparing Exercises

Load two exercises side by side to see how the graph grows
as vulnerabilities are introduced.

In [16]:
g_e01 = load(t.r01.e01)
g_e03 = load(t.r01.e03)

stats_e01 = g_e01.stats()
stats_e03 = g_e03.stats()

print(f"{'Metric':<20} {'e01':>8} {'e03':>8} {'delta':>8}")
print("-" * 48)
for key in stats_e01:
    v1 = stats_e01[key]
    v3 = stats_e03[key]
    delta = v3 - v1
    sign = "+" if delta > 0 else ""
    print(f"{key:<20} {v1:>8} {v3:>8} {sign}{delta:>7}")

print()
print("e01 endpoints:", [r.handler_name for r in g_e01.routes])
print("e03 endpoints:", [r.handler_name for r in g_e03.routes])
print("New in e03:", set(r.handler_name for r in g_e03.routes) - set(r.handler_name for r in g_e01.routes))

Metric                    e01      e03    delta
------------------------------------------------
nodes                      51       68 +     17
edges                      77      117 +     40
call_sites                 93      142 +     49
routes                      7       10 +      3
input_accesses             12       15 +      3
before_requests             0        0       0
dict_merges                 0        1 +      1
unique_keys                 6        7 +      1
unique_sources              3        3       0

e01 endpoints: ['index', 'view_balance', 'list_menu_items', 'list_orders', 'create_new_order', 'e2e_reset', 'e2e_balance']
e03 endpoints: ['index', 'view_balance', 'list_menu_items', 'list_orders', 'create_new_order', 'create_new_cart', 'add_item_to_cart_endpoint', 'checkout_cart', 'e2e_reset', 'e2e_balance']
New in e03: {'checkout_cart', 'create_new_cart', 'add_item_to_cart_endpoint'}
